In [ ]:
#%pip install pyperclip

# %pip install  pyproj

In [ ]:
import os
import pyperclip
from dotenv import load_dotenv, find_dotenv
from datetime import datetime, date

# Check .env file path and load environment variables

from pathlib import Path
load_dotenv(find_dotenv())

path_to_folder1 = os.environ.get('FOLDER_C1')
path_to_folder2 = os.environ.get('FOLDER_C2')
path_to_folder3 = os.environ.get('FOLDER_C3')

# Specify the path to the map document (aprx) you want to work with
aprx_path = os.environ.get('APRX')

# GDB for CSV-to-point conversion / GDB para conversión CSV a punto
gdb_path = os.environ.get('CSVPOINT_TO_GDB')

# Quarter prefix helper / Prefijo de trimestre
def current_quarter():
    today = date.today()
    q = (today.month - 1) // 3 + 1
    return f"T{today.year}_Q{q}"

coordinate_system_excel = 'PROJCS["GTM",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-90.5],PARAMETER["Scale_Factor",0.9998],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]];-5122000 -10000100 10000;-100000 10000;-100000 10000;0.001;0.001;0.001;IsHighPrecision'

In [ ]:
import re
import os
import pandas as pd
import pyproj
import arcpy
from datetime import datetime

def dms_to_dd(dms):
    if not isinstance(dms, str):
        raise ValueError(f"Input must be a string, got {type(dms)}: {dms}")
    # Regular expression patterns for DMS with and without a direction
    pattern_with_direction = re.compile(r"(\d+)[°]?\s*(\d+)'?\s*([\d.]+)\"?\s*([NSEWO]?)", re.IGNORECASE)
    pattern_without_direction = re.compile(r"(\d+)[°]?\s*(\d+)'?\s*([\d.]+)\"?", re.IGNORECASE)

    # Try matching with direction first
    match = pattern_with_direction.match(dms)
    if match:
        degrees, minutes, seconds, direction = match.groups()
        dd = float(degrees) + float(minutes) / 60 + float(seconds) / 3600
        if direction.upper() in ['S', 'W', 'O'] or (not direction and 88 <= float(degrees) <= 92):
            dd *= -1
        return dd
    else:
        match = pattern_without_direction.match(dms)
        if match:
            degrees, minutes, seconds = match.groups()
            dd = float(degrees) + float(minutes) / 60 + float(seconds) / 3600
            if 88 <= float(degrees) <= 92:
                dd *= -1
            return dd
    raise ValueError(f"Invalid DMS format: {dms}")

def safe_dms_to_dd(value):
    try:
        return dms_to_dd(value)
    except ValueError:
        return None

def convert_coordinate(coord):
    """
    If the coordinate is a string and contains a DMS indicator ('deg'),
    convert it using safe_dms_to_dd. Otherwise, attempt a direct float conversion.
    """
    if isinstance(coord, str) and "°" in coord:
        return safe_dms_to_dd(coord)
    try:
        return float(coord)
    except Exception:
        return None

def point_excel_to_shape(cdg, directory):
    # Process the CDG value and set up paths
    comp = cdg.split('_')[1]
    result_alphabet = re.sub(r"[^\w_]+", "", cdg)
    folder_path = os.path.join(directory, comp, cdg)
    files = os.listdir(folder_path)
    
    # Filter for Excel files containing area-related terms
    area_files = [
        file for file in files 
        if file.lower().endswith(('.xls', '.xlsx')) and 
           any(term in file.lower() for term in ['área', 'area', 'hectares', 'ha', 'has', '_reas_'])
    ]
    if not area_files:
        raise FileNotFoundError("No suitable Excel file found in the directory.")
    file_path = os.path.join(folder_path, area_files[0])
    
    # Read the second sheet of the Excel file
    df = pd.read_excel(file_path, sheet_name=1)
    
    # Clean and convert 'X' and 'Y' columns
    for col in ['X', 'Y']:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].str.replace(r"[^0-9°'\". NSEWO-]", "", regex=True)
        df[col] = df[col].apply(convert_coordinate)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(method='ffill')
    
    # Ensure 'Area (ha)' is numeric
    df['Área (ha)'] = pd.to_numeric(df['Área (ha)'], errors='coerce')
    required_columns = ['X', 'Y', 'Área (ha)', 'Beneficiario']
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"Required columns {required_columns} are missing in the data.")
    
    # Create a mask for valid rows
    valid_mask = df['Área (ha)'].notna() & df['Beneficiario'].notna()
    print("Forward-fill applied successfully.")
    
    # Decide if the coordinates are in WGS84 (lat/lon) or already numeric (GTM)
    mean_x = df['X'].mean()
    mean_y = df['Y'].mean()
    if 13 <= mean_x <= 18 and -92 <= mean_y <= -88:
        print("Coordinates appear to be in WGS84. Transforming to GTM.")
        transformer = pyproj.Transformer.from_crs("EPSG:4326", coordinate_system_excel, always_xy=True)
        df['X'], df['Y'] = transformer.transform(df['Y'].values, df['X'].values)
    else:
        # Check if swapping X and Y is needed (if values are reversed)
        if not (350000 <= mean_x <= 500000) or not (1500000 <= mean_y <= 1800000):
            if (350000 <= mean_y <= 500000) and (1500000 <= mean_x <= 1800000):
                df['X'], df['Y'] = df['Y'], df['X']
    
    df_valid = df[valid_mask]
    
    # Group by X and Y to calculate total area and beneficiary count
    if pd.to_numeric(df_valid['Beneficiario'], errors='coerce').notna().all():
        grouped = df_valid.groupby(['X', 'Y']).agg({
            'Área (ha)': 'sum',
            'Beneficiario': 'sum'
        }).reset_index()
    else:
        grouped = df_valid.groupby(['X', 'Y']).agg({
            'Área (ha)': 'sum',
            'Beneficiario': 'size'
        }).reset_index()
    
    grouped.rename(columns={'Beneficiario': 'num_bnf'}, inplace=True)
    grouped['CadaParcelas'] = grouped['Área (ha)'] / grouped['num_bnf']
    grouped['CdgActvdd'] = cdg
    grouped['Area_ha'] = grouped['Área (ha)']
    
    # Save the grouped data to a CSV file
    output_csv = os.path.join(folder_path, f"{cdg}.csv")
    grouped.to_csv(output_csv, encoding='utf-8-sig', index=False)
    
    # Create a point feature class: {quarter}_{codigo_de_la_actividad}
    # e.g. T2025_Q4_241014_C1_ROM_a
    arcpy.env.workspace = gdb_path
    qt = current_quarter()
    feature_class_name = f"{qt}_{result_alphabet}"
    arcpy.management.XYTableToPoint(
        in_table=output_csv,
        out_feature_class=os.path.join(gdb_path, feature_class_name),
        x_field="X",
        y_field="Y",
        coordinate_system=coordinate_system_excel
    )
    
    # Load the new feature class into the map
    aprx = arcpy.mp.ArcGISProject(aprx_path)
    map_obj = aprx.listMaps(os.environ.get('AR_MAP_NAME'))[0]
    map_obj.addDataFromPath(os.path.join(gdb_path, feature_class_name))
    print(f"Process complete. Data saved to {output_csv} and shapefile created.")
    print(f"Feature class: {feature_class_name} in {gdb_path}")

In [ ]:
if __name__ == "__main__":
    # Define the directory path (update if needed)
    directory = os.environ.get('SMARTSHEET_ATTACH_DIR')
    # Check if the directory is None and raise an error if it is
    if directory is None:
        raise ValueError("Environment variable 'ATTACH' is not set. Please check your .env file.")

    try:
        # Get clipboard content and split into lines
        clipboard_content = pyperclip.paste()
        cdg_list = [line.strip() for line in clipboard_content.split('\n') if line.strip()]
        
        if not cdg_list:
            print("No content found in clipboard. Please copy a list of CDG codes.")
            cdg = input("Input the code like '241014_C1_ROM_a': ")
            point_excel_to_shape(cdg=cdg, directory=directory)
        else:
            print(f"Found {len(cdg_list)} CDG codes in clipboard:")
            for i, cdg_code in enumerate(cdg_list, 1):
                print(f"{i}. {cdg_code}")
            
            print("\nProcessing options:")
            print("1. Process all codes automatically")
            print("2. Select specific codes to process")
            print("3. Process just one code")
            
            choice = input("Choose an option (1-3): ").strip()
            
            if choice == "1":
                # Process all codes
                for i, cdg in enumerate(cdg_list, 1):
                    print(f"\n{'='*60}")
                    print(f"Processing {i}/{len(cdg_list)}: {cdg}")
                    print('='*60)
                    try:
                        point_excel_to_shape(cdg=cdg, directory=directory)
                        print(f"✅ Successfully processed: {cdg}")
                    except Exception as e:
                        print(f"❌ Error processing {cdg}: {e}")
                        
            elif choice == "2":
                # Select multiple codes
                print("\nEnter the numbers of codes to process (e.g., '1,3,5' or '1-3,7'):")
                selection = input("Selection: ").strip()
                
                selected_indices = []
                try:
                    # Parse ranges and individual numbers
                    for part in selection.split(','):
                        part = part.strip()
                        if '-' in part:
                            start, end = map(int, part.split('-'))
                            selected_indices.extend(range(start-1, end))
                        else:
                            selected_indices.append(int(part)-1)
                    
                    selected_indices = [i for i in selected_indices if 0 <= i < len(cdg_list)]
                    
                    if selected_indices:
                        for i, idx in enumerate(selected_indices, 1):
                            cdg = cdg_list[idx]
                            print(f"\n{'='*60}")
                            print(f"Processing {i}/{len(selected_indices)}: {cdg}")
                            print('='*60)
                            try:
                                point_excel_to_shape(cdg=cdg, directory=directory)
                                print(f"✅ Successfully processed: {cdg}")
                            except Exception as e:
                                print(f"❌ Error processing {cdg}: {e}")
                    else:
                        print("No valid selections made.")
                except ValueError:
                    print("Invalid selection format.")
                    
            elif choice == "3":
                # Select one code
                while True:
                    try:
                        selection = int(input(f"Select a number (1-{len(cdg_list)}): ")) - 1
                        if 0 <= selection < len(cdg_list):
                            cdg = cdg_list[selection]
                            print(f"Selected: {cdg}")
                            point_excel_to_shape(cdg=cdg, directory=directory)
                            break
                        else:
                            print("Invalid selection. Please try again.")
                    except ValueError:
                        print("Please enter a valid number.")
            else:
                print("Invalid option selected.")
    
    except Exception as e:
        print(f"Error reading clipboard: {e}")
        cdg = input("Input the code like '241014_C1_ROM_a': ")
        point_excel_to_shape(cdg=cdg, directory=directory)

In [ ]:
# New implementation: Process CDG codes from clipboard
import pyperclip
import os

def run_clipboard_processing():
    """
    Read CDG codes from clipboard and process them using the point_excel_to_shape function
    """
    # Get the directory from environment variable
    directory = os.environ.get('ATTACH')
    
    # Check if the directory is set
    if directory is None:
        print("❌ Environment variable 'ATTACH' is not set. Please check your .env file.")
        return
    
    try:
        # Get clipboard content
        clipboard_content = pyperclip.paste()
        
        if not clipboard_content.strip():
            print("❌ No content found in clipboard. Please copy CDG codes to clipboard first.")
            return
        
        # Split clipboard content into lines and filter valid CDG codes
        cdg_list = [line.strip() for line in clipboard_content.split('\n') 
                   if line.strip() and not line.strip().startswith('#')]
        
        if not cdg_list:
            print("❌ No valid CDG codes found in clipboard.")
            return
        
        print(f"✅ Found {len(cdg_list)} CDG codes in clipboard:")
        for i, cdg_code in enumerate(cdg_list, 1):
            print(f"  {i}. {cdg_code}")
        
        print(f"\n📂 Using directory: {directory}")
        
        # Process each CDG code
        success_count = 0
        error_count = 0
        
        for i, cdg in enumerate(cdg_list, 1):
            print(f"\n{'='*60}")
            print(f"Processing {i}/{len(cdg_list)}: {cdg}")
            print('='*60)
            
            try:
                point_excel_to_shape(cdg=cdg, directory=directory)
                print(f"✅ Successfully processed: {cdg}")
                success_count += 1
            except Exception as e:
                print(f"❌ Error processing {cdg}: {e}")
                error_count += 1
        
        # Summary
        print(f"\n{'='*60}")
        print("PROCESSING SUMMARY")
        print('='*60)
        print(f"Total processed: {len(cdg_list)}")
        print(f"✅ Successful: {success_count}")
        print(f"❌ Failed: {error_count}")
        print('='*60)
        
    except Exception as e:
        print(f"❌ Error reading from clipboard: {e}")

# Run the function
if __name__ == "__main__":
    run_clipboard_processing()

## Test Clipboard Processing
Test how your clipboard content gets processed into a list before running the main function.

In [ ]:
# Test clipboard processing and create variables for later use
import pyperclip

# Paste your CDG list here and run this cell to see how it gets processed
clipboard_content = """
# Paste your CDG list between these triple quotes
# For example:
# 241014_C1_ROM_a
# 241015_C2_LOP_b
# 241016_C3_TEC_c
"""

print("Raw clipboard content:")
print(repr(clipboard_content))
print("\n" + "="*50 + "\n")

# Process the content the same way as in the main function
cdg_list = [line.strip() for line in clipboard_content.split('\n') if line.strip() and not line.strip().startswith('#')]

print(f"Processed CDG list ({len(cdg_list)} items):")
for i, cdg_code in enumerate(cdg_list, 1):
    print(f"{i}. {cdg_code}")

print("\n" + "="*50 + "\n")

# Test with actual clipboard content and create the final variable
try:
    actual_clipboard = pyperclip.paste()
    print("Actual clipboard content:")
    print(repr(actual_clipboard))
    
    # This is the variable you can use in other cells
    cdg_list_from_clipboard = [line.strip() for line in actual_clipboard.split('\n') if line.strip()]
    print(f"\nActual CDG list from clipboard ({len(cdg_list_from_clipboard)} items):")
    for i, cdg_code in enumerate(cdg_list_from_clipboard, 1):
        print(f"{i}. {cdg_code}")
    
    print(f"\n✅ Variable 'cdg_list_from_clipboard' created with {len(cdg_list_from_clipboard)} items")
    print("You can now use this variable in other cells!")
        
except Exception as e:
    print(f"Error reading actual clipboard: {e}")
    cdg_list_from_clipboard = []  # Empty list if clipboard fails

# Also create a variable from the manual input above (if you pasted anything there)
if cdg_list:
    print(f"\n✅ Variable 'cdg_list' created with {len(cdg_list)} items from manual input")
else:
    print(f"\n⚠️ Variable 'cdg_list' is empty (no manual input provided)")

print("\n" + "="*50)
print("AVAILABLE VARIABLES:")
print(f"• cdg_list: {len(cdg_list)} items (from manual input above)")
print(f"• cdg_list_from_clipboard: {len(cdg_list_from_clipboard) if 'cdg_list_from_clipboard' in locals() else 0} items (from actual clipboard)")
print("="*50)

## Use CDG List Variable
Process multiple CDG codes using the list from the test above.

In [ ]:
# Process multiple CDG codes using the variables from the test cell above
directory = os.environ.get('SsheetDataDirectory')

# Choose which list to use (make sure you ran the test cell first!)
# Option 1: Use clipboard content
try:
    active_cdg_list = cdg_list_from_clipboard
    print(f"Using clipboard list with {len(active_cdg_list)} items")
except NameError:
    print("⚠️ cdg_list_from_clipboard not found. Run the test cell first!")
    active_cdg_list = []

# Option 2: Use manual input (uncomment the line below to use this instead)
# active_cdg_list = cdg_list

if not active_cdg_list:
    print("No CDG list available. Please run the test cell first or check your clipboard content.")
else:
    print(f"\nProcessing {len(active_cdg_list)} CDG codes:")
    
    # Option A: Process all codes automatically
    process_all = input(f"Process all {len(active_cdg_list)} codes automatically? (y/n): ").lower().startswith('y')
    
    if process_all:
        for i, cdg in enumerate(active_cdg_list, 1):
            print(f"\n{'='*60}")
            print(f"Processing {i}/{len(active_cdg_list)}: {cdg}")
            print('='*60)
            try:
                point_excel_to_shape(cdg=cdg, directory=directory)
                print(f"✅ Successfully processed: {cdg}")
            except Exception as e:
                print(f"❌ Error processing {cdg}: {e}")
    else:
        # Option B: Select specific codes to process
        print("\nAvailable CDG codes:")
        for i, cdg_code in enumerate(active_cdg_list, 1):
            print(f"{i}. {cdg_code}")
        
        while True:
            try:
                choice = input(f"\nSelect a number (1-{len(active_cdg_list)}) or 'q' to quit: ")
                if choice.lower() == 'q':
                    break
                    
                index = int(choice) - 1
                if 0 <= index < len(active_cdg_list):
                    selected_cdg = active_cdg_list[index]
                    print(f"\nProcessing: {selected_cdg}")
                    try:
                        point_excel_to_shape(cdg=selected_cdg, directory=directory)
                        print(f"✅ Successfully processed: {selected_cdg}")
                    except Exception as e:
                        print(f"❌ Error processing {selected_cdg}: {e}")
                else:
                    print("Invalid selection. Please try again.")
            except ValueError:
                print("Please enter a valid number or 'q' to quit.")

## Old Version

In [ ]:

def dms_to_dd(dms):
    if not isinstance(dms, str):
        raise ValueError(f"Input must be a string, got {type(dms)}: {dms}")
    
    # Regular expression patterns for DMS
    pattern_with_direction = re.compile(r"(\d+)[°]?\s*(\d+)'?\s*([\d.]+)\"?\s*([NSEWO]?)", re.IGNORECASE)
    pattern_without_direction = re.compile(r"(\d+)[°]?\s*(\d+)'?\s*([\d.]+)\"?", re.IGNORECASE)

    # Try matching with direction
    match_with_direction = pattern_with_direction.match(dms)
    match_without_direction = pattern_without_direction.match(dms)

    if match_with_direction:
        degrees, minutes, seconds, direction = match_with_direction.groups()
        print(f"Matched with direction: {degrees}° {minutes}' {seconds}\" {direction}")
        dd = float(degrees) + float(minutes) / 60 + float(seconds) / (60 * 60)
        
        # If direction is present, handle accordingly
        if direction.upper() in ['S', 'W', 'O']:  # Negate for south or west
            dd *= -1
        # If direction is empty, infer based on value ranges
        elif direction == "":
            if 88 <= float(degrees) <= 92:  # Longitude (Y): Negate if within this range
                dd *= -1
        return dd

    elif match_without_direction:
        degrees, minutes, seconds = match_without_direction.groups()
        print(f"Matched without direction: {degrees}° {minutes}' {seconds}\"")
        dd = float(degrees) + float(minutes) / 60 + float(seconds) / (60 * 60)
        
        # Infer direction based on value ranges
        if 88 <= float(degrees) <= 92:  # Longitude (Y): Negate if within this range
            dd *= -1
        return dd

    else:
        print(f"Invalid DMS format: {dms}")  # Log the invalid input
        raise ValueError("Invalid DMS format")


def safe_dms_to_dd(value):
    try:
        return dms_to_dd(value)
    except ValueError:
        return None  # Mark invalid entries as None
    
def point_excel_to_shape(cdg, directory):
    # Process the 'cdg' value
    comp = cdg.split('_')[1]
    result_alphabet = re.sub(r"[^\w_]+", "", cdg)

    # Path to the directory where the Excel files are stored
    folder_path = os.path.join(directory, comp, cdg)

    # List all files in the directory
    files = os.listdir(folder_path)

    # Filter for Excel files containing area-related terms
    area_files = [
        file for file in files if file.lower().endswith(('.xls', '.xlsx')) and
        any(term in file.lower() for term in ['área', 'area', 'hectares', 'ha', 'has'])
    ]

    if not area_files:
        raise FileNotFoundError("No suitable Excel file found in the directory.")

    # Assume we take the first matched file
    file_path = os.path.join(folder_path, area_files[0])

    # Read the second sheet of the Excel file
    df__puntos_ha = pd.read_excel(file_path, sheet_name=1)
    
    # Ensure 'X' and 'Y' columns are cleaned and converted
    df__puntos_ha['X'] = df__puntos_ha['X'].astype(str).str.strip()  # Convert to string and remove spaces
    df__puntos_ha['X'] = df__puntos_ha['X'].str.replace(r"[^0-9°'\". NSEWO]", "", regex=True)
    df__puntos_ha['Y'] = df__puntos_ha['Y'].astype(str).str.strip()  # Convert to string and remove spaces
    df__puntos_ha['Y'] = df__puntos_ha['Y'].str.replace(r"[^0-9°'\". NSEWO]", "", regex=True)
    
    # Apply the conversion to 'X' and 'Y' columns
    if pd.api.types.is_string_dtype(df__puntos_ha['X']):
        df__puntos_ha['X'] = df__puntos_ha['X'].apply(safe_dms_to_dd)
    if pd.api.types.is_string_dtype(df__puntos_ha['Y']):
        df__puntos_ha['Y'] = df__puntos_ha['Y'].apply(safe_dms_to_dd)
    
    # Ensure 'X', 'Y', 'Área (ha)', and 'Beneficiario' columns exist
    if all(col in df__puntos_ha.columns for col in ['X', 'Y', 'Área (ha)', 'Beneficiario']):
        # Forward-fill X and Y coordinates for all rows
        df__puntos_ha['X'] = pd.to_numeric(df__puntos_ha['X'], errors='coerce').fillna(method='ffill')
        df__puntos_ha['Y'] = pd.to_numeric(df__puntos_ha['Y'], errors='coerce').fillna(method='ffill')

        # Ensure numeric conversion for 'Área (ha)'
        df__puntos_ha['Área (ha)'] = pd.to_numeric(df__puntos_ha['Área (ha)'], errors='coerce')

        # Create a mask for rows where 'Área (ha)' and 'Beneficiario' are not null
        valid_rows_mask = df__puntos_ha['Área (ha)'].notna() & df__puntos_ha['Beneficiario'].notna()

        print("Forward-fill applied successfully.")
    else:
        raise ValueError("Required columns ('X', 'Y', 'Área (ha)', 'Beneficiario') are missing in the data.")

    # Decide if the coordinates look like decimal degrees or GTM
    mean_x = df__puntos_ha['X'].mean()
    mean_y = df__puntos_ha['Y'].mean()

    # Check approximate latitude/longitude ranges for Guatemala
    # Latitude: 13 to 18, Longitude: -92 to -88
    if 13 <= mean_x <= 18 and -92 <= mean_y <= -88:
        print("Coordinates appear to be WGS 1984. Transforming to GTM.")
        transformer = pyproj.Transformer.from_crs("EPSG:4326", coordinate_system_excel, always_xy=True)
        df__puntos_ha['X'], df__puntos_ha['Y'] = transformer.transform(
            df__puntos_ha['Y'].values, df__puntos_ha['X'].values
        )
    else:
        # Otherwise, assume GTM and swap if needed
        if not (350000 <= mean_x <= 500000) or not (1500000 <= mean_y <= 1800000):
            if (350000 <= mean_y <= 500000) and (1500000 <= mean_x <= 1800000):
                df__puntos_ha['X'], df__puntos_ha['Y'] = df__puntos_ha['Y'], df__puntos_ha['X']

    # Apply the mask to filter only valid rows
    df_valid = df__puntos_ha[valid_rows_mask]

    # Group by X and Y to calculate total Area and count beneficiaries
    if pd.to_numeric(df_valid['Beneficiario'], errors='coerce').notna().all():
        # If Beneficiario contains numbers, use sum
        df__puntos_ha_grouped = df_valid.groupby(['X', 'Y']).agg({
            'Área (ha)': 'sum',
            'Beneficiario': 'sum'  # Sum the numbers in Beneficiario column
        }).reset_index()
    else:
        # If Beneficiario contains non-numeric values, count entries
        df__puntos_ha_grouped = df_valid.groupby(['X', 'Y']).agg({
            'Área (ha)': 'sum',
            'Beneficiario': 'size'  # Count all rows for each coordinate
        }).reset_index()


    # Rename the beneficiary count column to num_bnf
    df__puntos_ha_grouped.rename(columns={'Beneficiario': 'num_bnf'}, inplace=True)

    # Calculate area per beneficiary (CadaParcelas)
    df__puntos_ha_grouped['CadaParcelas'] = df__puntos_ha_grouped['Área (ha)'] / df__puntos_ha_grouped['num_bnf']

    # Add additional fields
    df__puntos_ha_grouped['CdgActvdd'] = cdg
    df__puntos_ha_grouped['Area_ha'] = df__puntos_ha_grouped['Área (ha)']

    # Save to CSV
    output_csv = os.path.join(folder_path, f"{cdg}.csv")
    df__puntos_ha_grouped.to_csv(output_csv, encoding='utf-8-sig', index=False)

    # Create a point feature class: {quarter}_{codigo_de_la_actividad}
    # e.g. T2025_Q4_241014_C1_ROM_a
    arcpy.env.workspace = gdb_path
    qt = current_quarter()
    feature_class_name = f"{qt}_{result_alphabet}"

    arcpy.management.XYTableToPoint(
        in_table=output_csv,
        out_feature_class=os.path.join(gdb_path, feature_class_name),
        x_field="X",
        y_field="Y",
        coordinate_system=coordinate_system_excel
    )

    # Load the new feature class into the map
    aprx = arcpy.mp.ArcGISProject(aprx_path)
    map_obj = aprx.listMaps(os.environ.get('AR_MAP_NAME'))[0]
    map_obj.addDataFromPath(os.path.join(gdb_path, feature_class_name))

    print(f"Process complete. Data saved to {output_csv} and shapefile created.")
    print(f"Feature class: {feature_class_name} in {gdb_path}")

directory = os.environ.get('SMARTSHEET_ATTACH_DIR')
point_excel_to_shape(cdg = input("Enter the CDG value (e.g., '231018_C1_LOP_c'): "), directory=directory )

In [ ]:
# import the GIS class in gis module
from arcgis.gis import GIS
import pandas as pd
gis = GIS('home')
print("Logged in as " + str(gis.properties.user.username))

feature_service_id = os.environ.get('AGOL_POLYGON_ITEM_ID')  # set in your .env file
content = gis.content.get(feature_service_id)
feature_lyr = content.layers[0]

# create a Spatially Enabled DataFrame object
df = pd.DataFrame.spatial.from_layer(feature_lyr)
df